# Lesson 7 : Agent Skills

Agent Skill is first introduced by Anthropic on October 2025 and now it's an open (and cross-platform) project.  
The objective of Agent Skills is to enable highly modularization and reuse of Agent skills and knowledge - such as, instructions, tool usage, and knowledge retrieval.

In this exercise, we create a brief custom skill to create a expense report, and we then use this our own skill with Agent Framework.

## Create a custom skill

There exist a lot of pre-built (reusable) existing skills (see [here](https://github.com/heilcheng/awesome-agent-skills)), but in this exercise, we briefly build a custom skill to create a unique expense report for our virtual company as follows.

First we create a file (named ```SKILL.md```) to describe skill's specification body.

In [1]:
import os
skill_folder = path = os.path.join("skills", "corporate-expense-report")
os.makedirs(skill_folder, exist_ok=True)

In [2]:
%%writefile skills/corporate-expense-report/SKILL.md
---
name: corporate-expense-report
description: This skill calculates tax and create a corporate expense report using template.
---

# Corporate Expense Report Skill

This skill provides instructions on how to create a corporate expense report.

## Capabilities

- Infer the category from the description in each expense item.  
  Available categories: "international transport", "domestic transport", "meal", "misc"
- Calculate tax and transaction fee using the following script in each items
- Generate a expense report including tax and transaction fee

## Scripts

- [assets/calculate_tax_and_trans.py](assets/calculate_tax_and_trans.py): Calculate tax and transaction fee (total) for each item. The input value of "category" should be one of "international transport", "domestic transport", "meal", or "misc". Please modify the category to match this input.

## Input Format

Provide the list of expense, which includes description, date, and amount.

## Output Format

Show expense report using the following format.

| Date | Category | Amount (USD) | Tax and transaction (USD) | Sub total (USD) |
|------|----------|--------------|---------------------------|-----------------|
|      |          |              |                           |                 |

Total: [total fee]

Writing skills/corporate-expense-report/SKILL.md


In above skill's specification, we refer an asset ```calculate_tax_and_trans.py``` which includes a function to calculate tax and transaction fee with my own company's policy.  
Now we create this asset (```calculate_tax_and_trans.py```) as follows.

In [3]:
asset_folder = path = os.path.join(skill_folder, "assets")
os.makedirs(asset_folder, exist_ok=True)

In [4]:
%%writefile skills/corporate-expense-report/assets/calculate_tax_and_trans.py
"""
Provides a function to calculate tax and transaction fee (total) for expense item.
"""

def calculate_tax_and_transaction_from_data(category: str, amount: int) -> int:
    """
    a function to calculate tax and transaction fee (total).

    Args:
        category:       One of items - "international transport", "domestic transport", "meal", or "misc"
        amount:         Dictionary with financial statement data

    Returns:
        Tax and transaction fee (total)
    """
    tax_ratio = 0.0
    transaction_ratio = 0.0

    if category.lower() == "international transport":
        tax_ratio = 0.00
        transaction_ratio = 0.10
    elif category.lower() == "domestic transport":
        tax_ratio = 0.00
        transaction_ratio = 0.05
    elif category.lower() == "meal":
        tax_ratio = 0.10
        transaction_ratio = 0.00
    elif category.lower() == "misc":
        tax_ratio = 0.03
        transaction_ratio = 0.00
    else:
        raise Exception(f"unsupported category: {category.lower()}")

    return int(amount * tax_ratio) + int(amount * transaction_ratio)

Writing skills/corporate-expense-report/assets/calculate_tax_and_trans.py


The ```skills``` folder will then have the following structure.  
In this exercise, we use only a single skill to solve the problem, but you can include a lot of existing skills in ```skills``` folder.

```
skills/
└── corporate-expense-report/
    ├── SKILL.md
    └── assets/
        └── calculate_tax_and_trans.py
```

## Run agent with skills

Now let's build an agent to use skills.

First we initilize the client object as usual.

In [5]:
from dotenv import load_dotenv
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = AzureAIClient(
    credential=credential,
)

Next we create an agent with skill provider as follows.  
Same as Lesson 6, we set a provider in ```context_providers``` property in the agent.]

In order to run a function in above skill's asset ```calculate_tax_and_trans.py```, we also set a code interpreter tool in this example.

In [6]:
from agent_framework import Agent, FileAgentSkillsProvider

skills_provider = FileAgentSkillsProvider(skill_paths="skills")
code_interpreter_tool = client.get_code_interpreter_tool()

agent = Agent(
    name="AgentWithSkills",
    client=client,
    instructions="You are a helpful assistant that can write and execute Python code to solve problems.",
    tools=[code_interpreter_tool],
    context_providers=[skills_provider],
)

Now let's run the agent.  
In this skill, the following ration of tax and transaction fee is added in each item, and the agent shows an expense report with a table format which columns are "data", "category", "amount", "tax and transaction fee", and "sub total".

| category | tax ratio | transaction fee ratio |
|----------|-----------|-----------------------|
| international transport | 0.00 | 0.10 |
| domestic transport | 0.00 | 0.05 |
| meal | 0.10 | 0.00 |
| misc | 0.03 | 0.00 |

In [7]:
from IPython.display import Markdown, display

prompt = """Return an expense report of the following.

- flight from JFK to SEA (round trip)
    - date : 2026/02/09
    - amount : 2800
- dinner
    - date : 2026/02/09
    - amount : 80
- lunch
    - date : 2026/02/10
    - amount : 37
- souvenirs
    - date : 2026/02/10
    - amount : 40
"""

result = await agent.run(prompt)
display(Markdown(result.text))

| Date | Category | Amount (USD) | Tax and transaction (USD) | Sub total (USD) |
|------|----------|--------------|---------------------------|-----------------|
| 2026/02/09 | domestic transport | 2800 | 140 | 2940 |
| 2026/02/09 | meal | 80 | 8 | 88 |
| 2026/02/10 | meal | 37 | 3 | 40 |
| 2026/02/10 | misc | 40 | 1 | 41 |

Total: 3109 USD

As you can see in the following message tracing, a function in above asset ```calculate_tax_and_trans.py``` is used and executed by using a code interpreter tool.

In [8]:
import agent_framework

for i, msg in enumerate(result.messages):
    print(f"********** message {i} **********")
    for c in msg.contents:
        if c.type == "function_call":
            print(f"*** {c.type} ***")
            print(f"call id : {c.call_id}")
            print(f"function name : {c.name}")
            print(f"function arguments : {c.arguments}")
        elif c.type == "function_result":
            print(f"*** {c.type} ***")
            print(f"call id : {c.call_id}")
            print(f"function result : {c.result}")
            print(f"exceptions : {c.exception}")
        elif c.type == "text":
            print(f"*** {c.type} ***")
            print(f"text : {c.text}")
        else:
            print(f"*** Other types : {c.type}***")

********** message 0 **********
*** function_call ***
call id : call_8npR92jvN08WY0ytSDT5Ogma
function name : load_skill
function arguments : {"skill_name":"corporate-expense-report"}
********** message 1 **********
*** function_result ***
call id : call_8npR92jvN08WY0ytSDT5Ogma
function result : # Corporate Expense Report Skill

This skill provides instructions on how to create a corporate expense report.

## Capabilities

- Infer the category from the description in each expense item.  
  Available categories: "international transport", "domestic transport", "meal", "misc"
- Calculate tax and transaction fee using the following script in each items
- Generate a expense report including tax and transaction fee

## Scripts

- [assets/calculate_tax_and_trans.py](assets/calculate_tax_and_trans.py): Calculate tax and transaction fee (total) for each item. The input value of "category" should be one of "international transport", "domestic transport", "meal", or "misc". Please modify the ca